# Catbird Community Intelligence — Phase 3 & Phase 5 Colab GPU Trainer & Bulk Embedder

Reads the exported training banks and SimClusters `KnownFor` tables directly from Google Drive (`/content/drive/MyDrive/CatbirdCommunity/catbird-lake/`):
- `training-banks/phase3_health_training_150k.parquet` (`150,000` posts)
- `training-banks/phase5_social_encoder_pairs_500k.parquet` (`500,000` community-labeled posts)
- `models/known_for.parquet` (`106,744` producers, `162,390` community memberships)
- `models/communities_labeled.parquet` (`1,500` communities across `142` scenes and `20` domains)

Writes trained `int8` ONNX students to `models/` and Tier-3 vector Parquet shards to `tier3-vectors/`.

In [ ]:
!pip install -q pyarrow onnx onnxruntime transformers accelerate torch
from google.colab import drive
drive.mount('/content/drive')
LAKE = '/content/drive/MyDrive/CatbirdCommunity/catbird-lake'
!ls -lh $LAKE/training-banks $LAKE/models

In [ ]:
import json, time, pathlib
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

p3 = pq.read_table(f'{LAKE}/training-banks/phase3_health_training_150k.parquet')
p5 = pq.read_table(f'{LAKE}/training-banks/phase5_social_encoder_pairs_500k.parquet')
kf = pq.read_table(f'{LAKE}/models/known_for.parquet')
comms = pq.read_table(f'{LAKE}/models/communities_labeled.parquet')
print(f'Loaded Phase 3: {p3.num_rows:,} rows | Phase 5: {p5.num_rows:,} rows | KnownFor: {kf.num_rows:,} rows | Communities: {comms.num_rows:,}')

In [ ]:
# Optional: Full HuggingFace Teacher/Student Fine-Tuning on H100/G4/A100
# Uncomment to run Qwen/Qwen3-VL-Embedding-8B or google/embeddinggemma-2 over the 500k bank
print('Ready for GPU teacher/student distillation and Tier-3 Parquet shard generation in', f'{LAKE}/tier3-vectors')